## Import Libraries

In [1]:
import os
os.chdir(r"..\website")

In [2]:
import pandas as pd
import requests
import re
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

from models import Database

db = Database(
    host=os.environ["HOST"],
    port=os.environ["PORT"],
    database=os.environ["DATABASE"],
    user=os.environ["USER"],
    password=os.environ["PASSWORD"]
    )

## Get Market Price

In [3]:
# Wah Chan Gold Price Website
url = "https://wahchan.com.my/gold-price/"

html = requests.get(url).text

match = re.search(r'916 Gold.*?RM\s*([\d,]+)', html, re.DOTALL)
if match:
    market_price = float(match.group(1).replace(',', ''))
    print(market_price)
else:
    print("Failed to locate the 916 Gold price.")

510.0


## Get report using SQL

In [4]:

query = f"""
WITH variables AS (
    SELECT {market_price} AS gold_price
)
SELECT 
    s.stk_id,
    s.stk_weight,
    s.stk_labor_cost,
    s.stk_size,
    ROUND(((s.stk_weight * v.gold_price) + (s.stk_labor_cost * 3.5)) * 1.1) AS shopee_price,
    s.stk_pattern,
    s.stk_tag,
    p.pur_code 
FROM konghin.stock s
LEFT JOIN konghin.purchase p ON s.stk_pur_id = p.pur_id,
     variables v
WHERE UPPER(s.stk_pattern) = 'KETUM';
"""

rows = db.select_raw(query)

if rows is not None:
    print(f"fetched {len(rows)} line of data")
else:
    print("fetched failed")


fetched 90 line of data


In [ ]:
report_name = 'KETUM'
path = r'C:\Users\keong\OneDrive\E-commerce\Stocks\Shopee stocks\SQL Output'

now = datetime.now()
formatted = now.strftime("%d_%m_%y@%H_%M_%S")

rows.to_excel(fr"{path}\{report_name}_{formatted}.xlsx",index = False)